# Voice Studio — mở trực tiếp, không cần tải file

**Cách dùng:**
1. Chọn **Runtime → Change runtime type → T4 GPU → Save**.
2. Chọn **Runtime → Run all** (Chạy tất cả). Nếu Colab hỏi, cho phép chạy notebook sau khi xem nguồn.
3. Chờ cài thư viện và mở giao diện Voice Studio ở ô cuối.
4. Chọn OmniVoice, tải mẫu MP3/WAV/FLAC 3–10 giây, nhập lời mẫu và văn bản cần đọc, rồi bấm **Tạo âm thanh**.

Không cần API key. GPU Colab miễn phí tùy hạn mức; lần đầu tải thư viện/model có thể mất nhiều phút. Tải WAV trước khi runtime hết hạn. Chỉ clone giọng của bạn hoặc người đã đồng ý.

Mặc định dùng OmniVoice cho tiếng Việt. Có thể chọn Piper trong danh sách engine của ô thứ hai để đọc bằng giọng có sẵn trên CPU. Coqui XTTS và Chatterbox trong cấu hình này không hỗ trợ tiếng Việt. Clone thật và runtime Colab chưa được kiểm thử trên GPU; Piper đã sinh WAV tiếng Việt trên CPU.


In [ ]:
# @title 1. Lấy Voice Studio từ GitHub
from pathlib import Path
import subprocess, sys, time, urllib.request
root = Path('/content/voice-studio')
repository = 'https://github.com/lsmedia2105/lsmedia.git'
if not root.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'main', repository, str(root)])
elif (root / '.git').is_dir() and (root / 'server.py').is_file():
    changes = subprocess.check_output(['git', '-C', str(root), 'status', '--porcelain'], text=True)
    if changes.strip():
        raise RuntimeError('Có thay đổi source trong runtime; không tự ghi đè. Lưu thay đổi trước khi cập nhật.')
    shallow = subprocess.check_output(['git', '-C', str(root), 'rev-parse', '--is-shallow-repository'], text=True).strip()
    fetch = ['git', '-C', str(root), 'fetch']
    if shallow == 'true':
        fetch.append('--unshallow')
    subprocess.check_call(fetch + ['origin', 'main'])
    subprocess.check_call(['git', '-C', str(root), 'merge', '--ff-only', 'origin/main'])
    print('Voice Studio đã được cập nhật từ GitHub.')
else:
    raise RuntimeError('Thư mục /content/voice-studio đã tồn tại nhưng không phải Voice Studio. Chọn Runtime → Disconnect and delete runtime rồi chạy lại.')
print('Voice Studio đã tải xong. Không cần ZIP.')


In [ ]:
# @title 2. Cài engine miễn phí
engine = 'omnivoice' # @param ['omnivoice', 'piper', 'coqui', 'chatterbox']
if engine not in ('omnivoice', 'piper', 'coqui', 'chatterbox'):
    raise ValueError('Engine không hợp lệ')
if engine != 'piper':
    try:
        gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
    except FileNotFoundError:
        gpu = None
    if gpu is None or gpu.returncode != 0 or not gpu.stdout.strip():
        raise RuntimeError('Chưa có GPU. Chọn Runtime → Change runtime type → T4 GPU → Save rồi chạy lại. Nếu hết hạn mức miễn phí, thử sau hoặc chọn Piper để đọc trên CPU.')
    print('GPU:', gpu.stdout.strip())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(root / 'requirements.txt'), 'librosa==0.11.0'])
script = 'prepare_piper.py' if engine == 'piper' else 'install_engine.py'
command = [sys.executable, str(root / script)]
if engine != 'piper':
    command.append(engine)
# Merge stderr into stdout so Colab displays the actual installation error.
from collections import deque
last_lines = deque(maxlen=35)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
    last_lines.append(line)
process.stdout.close()
if process.wait() != 0:
    raise RuntimeError('Cài engine thất bại. Chi tiết bên dưới:\n' + ''.join(last_lines))
print('Đã cài engine:', engine)


In [ ]:
# @title 3. Mở giao diện Voice Studio
from google.colab import output
if 'studio_process' in globals() and studio_process.poll() is None:
    print('Khởi động lại server để nhận phiên bản mới…')
    studio_process.terminate()
    try:
        studio_process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        studio_process.kill()
        studio_process.wait()
studio_log = open('/content/voice-studio-server.log', 'w')
studio_process = subprocess.Popen([sys.executable, str(root / 'server.py')], cwd=root, stdout=studio_log, stderr=studio_log)
for attempt in range(60):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/status', timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        if studio_process.poll() is not None:
            raise RuntimeError(Path('/content/voice-studio-server.log').read_text()[-3000:])
        time.sleep(1)
else:
    raise RuntimeError('Server chưa sẵn sàng; xem /content/voice-studio-server.log')
print('Giao diện sẽ hiện bên dưới. Nếu trống, chạy lại riêng ô này.')
output.serve_kernel_port_as_iframe(8000, height=1100)
